# download dataset

In [ ]:
# https://drive.google.com/file/d/1PFq4Ub19NXfPxOmEYQ_KiNoVFkZlTJSt/view?usp=sharing
!gdown --id 1PFq4Ub19NXfPxOmEYQ_KiNoVFkZlTJSt
!unzip /kaggle/working/diagram2graph_dataset.zip

# Zero Shot prompting

In [ ]:

import base64
import json
import os
import time
from pathlib import Path
from typing import Optional
import requests
from PIL import Image

# =========================
# 1) CONFIGURATION
# =========================
# Prefer loading your key from an environment variable (more secure).
# export GEMINI_API_KEY="..."
def load_api_key(name, fallback_names=()):
    import os
    import sys
    from pathlib import Path

    candidates = (name, *fallback_names)
    repo_root = next(
        (
            candidate
            for candidate in (Path.cwd(), *Path.cwd().parents)
            if (candidate / "common" / "__init__.py").exists()
        ),
        None,
    )

    get_api_key = None
    if repo_root is not None:
        if str(repo_root) not in sys.path:
            sys.path.insert(0, str(repo_root))
        try:
            from common import get_api_key as shared_get_api_key
        except ImportError:
            pass
        else:
            get_api_key = shared_get_api_key

    if get_api_key is not None:
        for candidate in candidates:
            try:
                return get_api_key(candidate)
            except KeyError:
                pass

    for candidate in candidates:
        value = os.getenv(candidate, "").strip()
        if value:
            return value

    joined = ", ".join(candidates)
    raise RuntimeError(
        f"Missing API key. Set one of [{joined}] in the environment"
        " or the top-level config file."
    )

GEMINI_API_KEY = load_api_key("GEMINI_API_KEY")

MODEL = "gemini-2.5-flash"
ENDPOINT = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent?key={GEMINI_API_KEY}"

# Input images directory (recursively scanned)
DATASET_DIR = "/kaggle/input/diagram2graph-dataset/diagram2graph"  # <--- change this
# Output directory for per-image TTL files
OUTPUT_DIR = "/kaggle/working/zeroshot_outputs"  # <--- change this
# Optional: also write a small manifest JSON with image->ttl mapping
WRITE_MANIFEST = True
MANIFEST_PATH = os.path.join(OUTPUT_DIR, "manifest.json")

VALID_IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}

# =========================
# 2) PROMPT (Zero-shot)
# =========================
PROMPT_TEXT = r"""

System
You are a diagram-to-graph extractor. Given a flowchart/diagram image, output ONLY valid Turtle using the diagram2graph (d2g) vocabulary.

Mandatory Instructions (follow exactly)
1) Prefixes (only these two)
@prefix d2g:  <http://example.org/diagram2graph#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

2) Identifiers (IRIs)
- Fixed patterns:
  Nodes: <http://example.org/diagram/diagram/node/{N}>
  Edges: <http://example.org/diagram/diagram/edge/{E}>
- Node numbering: visual reading order (top→bottom, left→right).
- Edge IDs (strict rule): build each edge IRI as ST, where S is the source node ID and T is the target node ID.
  Example: the edge from node/1 to node/2 MUST be <http://example.org/diagram/diagram/edge/12>

3) Nodes (create one node per readable shape)
For each shape:
<…/node/N> a d2g:Node, d2g:{Start|Process|Decision|Delay|Terminator} ;
    rdfs:label "exact text from the shape" ;
    d2g:shape d2g:{TartEvent|TaK|Gateway|Delay|EndEvent} .
Required shape mapping (use these exact dataset tokens—even if they look misspelled):
- Start/Begin (rounded oval) → d2g:TartEvent
- End/Stop   (rounded oval) → d2g:EndEvent
- Rectangle  (process)      → d2g:TaK
- Diamond    (decision)     → d2g:Gateway
- Delay      (delay symbol) → d2g:Delay

4) Edges (create a dedicated Edge resource per arrow)
For each arrow:
<…/edge/ST> a d2g:Edge, d2g:{Solid|Dashed} ;
    d2g:source <…/node/S> ;
    d2g:target <…/node/T> ;
    d2g:relationshipType d2g:{Follows|Branches} ;
    [optional] d2g:relationshipValue "label on the arrow (e.g., Yes/No)" .
Rules:
- Outgoing arrows from a Decision node must use relationshipType = d2g:Branches, and include relationshipValue if text exists (Yes/No/etc.).
- All other arrows use relationshipType = d2g:Follows.
- Set line style precisely: d2g:Solid for solid lines; d2g:Dashed for dotted/dashed lines.

5) Convenience node-to-node triples (also required)
- For Follows:  <…/node/S> d2g:follows  <…/node/T> .
- For Branches: <…/node/S> d2g:branches <…/node/T> .

6) Output rules
- Output Turtle only; no explanations or extra text.
- Group by subject; one triple per line; end every line with a period.
- Every edge’s source/target must reference existing node IRIs.
- Do NOT use any prefixes or vocabularies other than d2g and rdfs.
- Write class and shape tokens EXACTLY as specified above (no spelling changes).

"""

# =========================
# 3) HELPERS
# =========================
def get_mime_type(path: str) -> str:
    ext = Path(path).suffix.lower()
    if ext in {".jpg", ".jpeg"}:
        return "image/jpeg"
    if ext == ".png":
        return "image/png"
    if ext in {".tif", ".tiff"}:
        return "image/tiff"
    if ext == ".webp":
        return "image/webp"
    if ext == ".bmp":
        return "image/bmp"
    return "image/jpeg"

def encode_image_to_base64(image_path: str) -> Optional[str]:
    try:
        # Ensure the image is readable; PIL open also validates file integrity
        with Image.open(image_path) as _:
            pass
        data = Path(image_path).read_bytes()
        return base64.b64encode(data).decode("utf-8")
    except FileNotFoundError:
        print(f"[WARN] File not found: {image_path}")
        return None
    except Exception as e:
        print(f"[WARN] Failed to read {image_path}: {e}")
        return None

def build_request(image_b64: str, mime_type: str, prompt: str) -> dict:
    return {
        "contents": [
            {
                "role": "user",
                "parts": [
                    {"inlinedata": {"mimeType": mime_type, "data": image_b64}},
                    {"text": prompt},
                ],
            }
        ],
        "generationConfig": {
            "temperature": 0.1,
            "topP": 0.9,
            "maxOutputTokens": 8192,
        },
    }

def _strip_code_fences(text: str) -> str:
    """
    Removes ```turtle ... ``` or ``` ... ``` fences if present.
    """
    t = text.strip()
    if t.startswith("```"):
        first_newline = t.find("\n")
        if first_newline != -1 and t[:first_newline].startswith("```"):
            t = t[first_newline + 1 :]
        if t.endswith("```"):
            t = t[:-3].rstrip()
    return t.strip()

def call_gemini(payload: dict, max_retries: int = 4) -> Optional[str]:
    """
    Calls the Gemini API with basic retry/backoff on 429/5xx.
    Returns clean Turtle (TTL) text, or None.
    """
    headers = {"Content-Type": "application/json"}
    backoff = 2.0
    for attempt in range(max_retries):
        try:
            resp = requests.post(ENDPOINT, headers=headers, data=json.dumps(payload), timeout=120)
            if resp.status_code >= 500 or resp.status_code == 429:
                print(f"[INFO] Retryable HTTP {resp.status_code}; attempt {attempt+1}/{max_retries}")
                time.sleep(backoff)
                backoff *= 1.8
                continue
            resp.raise_for_status()
            data = resp.json()
            if "candidates" in data and data["candidates"]:
                raw = data["candidates"][0]["content"]["parts"][0]["text"]
                return _strip_code_fences(raw)
            return None
        except requests.exceptions.RequestException as e:
            print(f"[WARN] Request error: {e}; attempt {attempt+1}/{max_retries}")
            time.sleep(backoff)
            backoff *= 1.8
    return None

def ensure_dir(path: str):
    Path(path).mkdir(parents=True, exist_ok=True)

# =========================
# 4) MAIN: per-image .ttl files
# =========================
def main():
    ensure_dir(OUTPUT_DIR)

    manifest = []  # optional record of what we wrote

    for root, _, files in os.walk(DATASET_DIR):
        for fname in files:
            ext = Path(fname).suffix.lower()
            if ext not in VALID_IMAGE_EXTS:
                continue

            img_path = os.path.join(root, fname)
            stem = Path(fname).stem  # "1.png" -> "1"
            ttl_path = os.path.join(OUTPUT_DIR, f"{stem}.ttl")

            print(f"[INFO] Processing: {img_path} -> {ttl_path}")

            image_b64 = encode_image_to_base64(img_path)
            if not image_b64:
                print(f"[SKIP] Could not encode {img_path}")
                continue

            mime = get_mime_type(img_path)
            payload = build_request(image_b64, mime, PROMPT_TEXT)
            ttl_text = call_gemini(payload)

            if not ttl_text:
                print(f"[WARN] No TTL returned for {img_path}")
                continue

            # Write the TTL file
            with open(ttl_path, "w", encoding="utf-8") as f:
                f.write(ttl_text if ttl_text.endswith("\n") else ttl_text + "\n")

            manifest.append({"source_image": img_path, "ttl_file": ttl_path})

    if WRITE_MANIFEST:
        ensure_dir(OUTPUT_DIR)
        with open(MANIFEST_PATH, "w", encoding="utf-8") as mf:
            json.dump({"items": manifest}, mf, indent=2, ensure_ascii=False)

    print(f"[DONE] Wrote {len(manifest)} TTL files to: {OUTPUT_DIR}")
    if WRITE_MANIFEST:
        print(f"[INFO] Manifest: {MANIFEST_PATH}")

if __name__ == "__main__":
    main()


# Zip output

In [ ]:
!zip -r /kaggle/working/Zeroshot_output.zip /kaggle/working/zeroshot_outputs

# oneshot Prompting

In [ ]:

import base64
import json
import os
import time
from pathlib import Path
from typing import Optional
import requests
from PIL import Image

# =========================
# 1) CONFIGURATION
# =========================
# Prefer loading your key from an environment variable (more secure).
# export GEMINI_API_KEY="..."
def load_api_key(name, fallback_names=()):
    import os
    import sys
    from pathlib import Path

    candidates = (name, *fallback_names)
    repo_root = next(
        (
            candidate
            for candidate in (Path.cwd(), *Path.cwd().parents)
            if (candidate / "common" / "__init__.py").exists()
        ),
        None,
    )

    get_api_key = None
    if repo_root is not None:
        if str(repo_root) not in sys.path:
            sys.path.insert(0, str(repo_root))
        try:
            from common import get_api_key as shared_get_api_key
        except ImportError:
            pass
        else:
            get_api_key = shared_get_api_key

    if get_api_key is not None:
        for candidate in candidates:
            try:
                return get_api_key(candidate)
            except KeyError:
                pass

    for candidate in candidates:
        value = os.getenv(candidate, "").strip()
        if value:
            return value

    joined = ", ".join(candidates)
    raise RuntimeError(
        f"Missing API key. Set one of [{joined}] in the environment"
        " or the top-level config file."
    )

GEMINI_API_KEY = load_api_key("GEMINI_API_KEY")

MODEL = "gemini-2.5-flash"
ENDPOINT = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent?key={GEMINI_API_KEY}"

# Input images directory (recursively scanned)
DATASET_DIR = "/kaggle/input/diagram2graph-dataset/diagram2graph"  # <--- change this
# Output directory for per-image TTL files
OUTPUT_DIR = "/kaggle/working/ttl_Onshot_outputs3"  # <--- change this
# Optional: also write a small manifest JSON with image->ttl mapping
WRITE_MANIFEST = True
MANIFEST_PATH = os.path.join(OUTPUT_DIR, "manifest.json")

VALID_IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}

# =========================
# 2) PROMPT (one-shot)
# =========================
PROMPT_TEXT = r"""

System
You are a diagram-to-graph extractor. Given a flowchart/diagram image, output ONLY valid Turtle using the diagram2graph (d2g) vocabulary.

Mandatory Instructions (follow exactly)
1) Prefixes (only these two)
@prefix d2g:  <http://example.org/diagram2graph#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

2) Identifiers (IRIs)
- Fixed patterns:
  Nodes: <http://example.org/diagram/diagram/node/{N}>
  Edges: <http://example.org/diagram/diagram/edge/{E}>
- Node numbering: visual reading order (top→bottom, left→right).
- Edge IDs (strict rule): build each edge IRI as ST, where S is the source node ID and T is the target node ID.
  Example: the edge between node/1 and node/2 MUST be <http://example.org/diagram/diagram/edge/12>

3) Nodes (create one node per readable shape)
For each shape:
<…/node/N> a d2g:Node, d2g:{Start|Process|Decision|Delay|Terminator} ;
    rdfs:label "exact text from the shape" ;
    d2g:shape d2g:{TartEvent|TaK|Gateway|Delay|EndEvent} .
Required shape mapping (use these exact dataset tokens—even if they look misspelled):
- Start/Begin (rounded oval) → d2g:TartEvent
- End/Stop   (rounded oval) → d2g:EndEvent
- Rectangle  (process)      → d2g:TaK
- Diamond    (decision)     → d2g:Gateway
- Delay      (delay symbol) → d2g:Delay

4) Edges (create a dedicated Edge resource per arrow)
For each arrow:
<…/edge/ST> a d2g:Edge, d2g:{Solid|Dashed} ;
    d2g:source <…/node/S> ;
    d2g:target <…/node/T> ;
    d2g:relationshipType d2g:{Follows|Branches} ;
    [optional] d2g:relationshipValue "label on the arrow (e.g., Yes/No)" .
Rules:
- Outgoing arrows from a Decision node must use relationshipType = d2g:Branches, and include relationshipValue if text exists (Yes/No/etc.).
- All other arrows use relationshipType = d2g:Follows.
- Set line style precisely: d2g:Solid for solid lines; d2g:Dashed for dotted/dashed lines.

5) Convenience node-to-node triples (also required)
- For Follows:  <…/node/S> d2g:follows  <…/node/T> .
- For Branches: <…/node/S> d2g:branches <…/node/T> .

6) Output rules
- Output Turtle only; no explanations or extra text.
- Group by subject; one triple per line; end every line with a period.
- Every edge’s source/target must reference existing node IRIs.
- Do NOT use any prefixes or vocabularies other than d2g and rdfs.
- Write class and shape tokens EXACTLY as specified above (no spelling changes).

One-shot Example (complete example obeying edge-ID ST rule and exact shape tokens)

@prefix d2g:  <http://example.org/diagram2graph#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

<http://example.org/diagram/diagram/node/1> a d2g:Start, d2g:Node ;
    rdfs:label "Start" ;
    d2g:shape d2g:TartEvent .

<http://example.org/diagram/diagram/node/2> a d2g:Process, d2g:Node ;
    rdfs:label "Initialize" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/3> a d2g:Decision, d2g:Node ;
    rdfs:label "Decision" ;
    d2g:shape d2g:Gateway .

<http://example.org/diagram/diagram/node/4> a d2g:Delay, d2g:Node ;
    rdfs:label "Delay" ;
    d2g:shape d2g:Delay .

<http://example.org/diagram/diagram/node/5> a d2g:Process, d2g:Node ;
    rdfs:label "Process" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/6> a d2g:Process, d2g:Node ;
    rdfs:label "Print result" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/7> a d2g:Terminator, d2g:Node ;
    rdfs:label "End" ;
    d2g:shape d2g:EndEvent .

<http://example.org/diagram/diagram/edge/12> a d2g:Edge, d2g:Solid ;
    d2g:source <http://example.org/diagram/diagram/node/1> ;
    d2g:target <http://example.org/diagram/diagram/node/2> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/edge/23> a d2g:Edge, d2g:Solid ;
    d2g:source <http://example.org/diagram/diagram/node/2> ;
    d2g:target <http://example.org/diagram/diagram/node/3> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/edge/36> a d2g:Edge, d2g:Solid ;
    d2g:source <http://example.org/diagram/diagram/node/3> ;
    d2g:target <http://example.org/diagram/diagram/node/6> ;
    d2g:relationshipType d2g:Branches ;
    d2g:relationshipValue "Yes" .

<http://example.org/diagram/diagram/edge/34> a d2g:Edge, d2g:Dashed ;
    d2g:source <http://example.org/diagram/diagram/node/3> ;
    d2g:target <http://example.org/diagram/diagram/node/5> ;
    d2g:relationshipType d2g:Branches ;
    d2g:relationshipValue "No" .

<http://example.org/diagram/diagram/edge/45> a d2g:Edge, d2g:Dashed ;
    d2g:source <http://example.org/diagram/diagram/node/5> ;
    d2g:target <http://example.org/diagram/diagram/node/4> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/edge/52> a d2g:Edge, d2g:Dashed ;
    d2g:source <http://example.org/diagram/diagram/node/4> ;
    d2g:target <http://example.org/diagram/diagram/node/2> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/edge/67> a d2g:Edge, d2g:Solid ;
    d2g:source <http://example.org/diagram/diagram/node/6> ;
    d2g:target <http://example.org/diagram/diagram/node/7> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/1> d2g:follows <http://example.org/diagram/diagram/node/2> .
<http://example.org/diagram/diagram/node/2> d2g:follows <http://example.org/diagram/diagram/node/3> .
<http://example.org/diagram/diagram/node/3> d2g:branches <http://example.org/diagram/diagram/node/6> .
<http://example.org/diagram/diagram/node/3> d2g:branches <http://example.org/diagram/diagram/node/5> .
<http://example.org/diagram/diagram/node/5> d2g:follows <http://example.org/diagram/diagram/node/4> .
<http://example.org/diagram/diagram/node/4> d2g:follows <http://example.org/diagram/diagram/node/2> .
<http://example.org/diagram/diagram/node/6> d2g:follows <http://example.org/diagram/diagram/node/7> .

"""

# =========================
# 3) HELPERS
# =========================
def get_mime_type(path: str) -> str:
    ext = Path(path).suffix.lower()
    if ext in {".jpg", ".jpeg"}:
        return "image/jpeg"
    if ext == ".png":
        return "image/png"
    if ext in {".tif", ".tiff"}:
        return "image/tiff"
    if ext == ".webp":
        return "image/webp"
    if ext == ".bmp":
        return "image/bmp"
    return "image/jpeg"

def encode_image_to_base64(image_path: str) -> Optional[str]:
    try:
        # Ensure the image is readable; PIL open also validates file integrity
        with Image.open(image_path) as _:
            pass
        data = Path(image_path).read_bytes()
        return base64.b64encode(data).decode("utf-8")
    except FileNotFoundError:
        print(f"[WARN] File not found: {image_path}")
        return None
    except Exception as e:
        print(f"[WARN] Failed to read {image_path}: {e}")
        return None

def build_request(image_b64: str, mime_type: str, prompt: str) -> dict:
    return {
        "contents": [
            {
                "role": "user",
                "parts": [
                    {"inlinedata": {"mimeType": mime_type, "data": image_b64}},
                    {"text": prompt},
                ],
            }
        ],
        "generationConfig": {
            "temperature": 0.1,
            "topP": 0.9,
            "maxOutputTokens": 8192,
        },
    }

def _strip_code_fences(text: str) -> str:
    """
    Removes ```turtle ... ``` or ``` ... ``` fences if present.
    """
    t = text.strip()
    if t.startswith("```"):
        first_newline = t.find("\n")
        if first_newline != -1 and t[:first_newline].startswith("```"):
            t = t[first_newline + 1 :]
        if t.endswith("```"):
            t = t[:-3].rstrip()
    return t.strip()

def call_gemini(payload: dict, max_retries: int = 4) -> Optional[str]:
    """
    Calls the Gemini API with basic retry/backoff on 429/5xx.
    Returns clean Turtle (TTL) text, or None.
    """
    headers = {"Content-Type": "application/json"}
    backoff = 2.0
    for attempt in range(max_retries):
        try:
            resp = requests.post(ENDPOINT, headers=headers, data=json.dumps(payload), timeout=120)
            if resp.status_code >= 500 or resp.status_code == 429:
                print(f"[INFO] Retryable HTTP {resp.status_code}; attempt {attempt+1}/{max_retries}")
                time.sleep(backoff)
                backoff *= 1.8
                continue
            resp.raise_for_status()
            data = resp.json()
            if "candidates" in data and data["candidates"]:
                raw = data["candidates"][0]["content"]["parts"][0]["text"]
                return _strip_code_fences(raw)
            return None
        except requests.exceptions.RequestException as e:
            print(f"[WARN] Request error: {e}; attempt {attempt+1}/{max_retries}")
            time.sleep(backoff)
            backoff *= 1.8
    return None

def ensure_dir(path: str):
    Path(path).mkdir(parents=True, exist_ok=True)

# =========================
# 4) MAIN: per-image .ttl files
# =========================
def main():
    ensure_dir(OUTPUT_DIR)

    manifest = []  # optional record of what we wrote

    for root, _, files in os.walk(DATASET_DIR):
        for fname in files:
            ext = Path(fname).suffix.lower()
            if ext not in VALID_IMAGE_EXTS:
                continue

            img_path = os.path.join(root, fname)
            stem = Path(fname).stem  # "1.png" -> "1"
            ttl_path = os.path.join(OUTPUT_DIR, f"{stem}.ttl")

            print(f"[INFO] Processing: {img_path} -> {ttl_path}")

            image_b64 = encode_image_to_base64(img_path)
            if not image_b64:
                print(f"[SKIP] Could not encode {img_path}")
                continue

            mime = get_mime_type(img_path)
            payload = build_request(image_b64, mime, PROMPT_TEXT)
            ttl_text = call_gemini(payload)

            if not ttl_text:
                print(f"[WARN] No TTL returned for {img_path}")
                continue

            # Write the TTL file
            with open(ttl_path, "w", encoding="utf-8") as f:
                f.write(ttl_text if ttl_text.endswith("\n") else ttl_text + "\n")

            manifest.append({"source_image": img_path, "ttl_file": ttl_path})

    if WRITE_MANIFEST:
        ensure_dir(OUTPUT_DIR)
        with open(MANIFEST_PATH, "w", encoding="utf-8") as mf:
            json.dump({"items": manifest}, mf, indent=2, ensure_ascii=False)

    print(f"[DONE] Wrote {len(manifest)} TTL files to: {OUTPUT_DIR}")
    if WRITE_MANIFEST:
        print(f"[INFO] Manifest: {MANIFEST_PATH}")

if __name__ == "__main__":
    main()


In [ ]:
!zip -r /kaggle/working/oneshot_outputs_v3.zip /kaggle/working/ttl_Onshot_outputs3

In [ ]:
!zip -r /kaggle/working/Onshot_outputs.zip /kaggle/working/ttl_Onshot_outputs2

# fewshot Prompting

In [ ]:

import base64
import json
import os
import time
from pathlib import Path
from typing import Optional
import requests
from PIL import Image

# =========================
# 1) CONFIGURATION
# =========================
# Prefer loading your key from an environment variable (more secure).
# export GEMINI_API_KEY="..."
def load_api_key(name, fallback_names=()):
    import os
    import sys
    from pathlib import Path

    candidates = (name, *fallback_names)
    repo_root = next(
        (
            candidate
            for candidate in (Path.cwd(), *Path.cwd().parents)
            if (candidate / "common" / "__init__.py").exists()
        ),
        None,
    )

    get_api_key = None
    if repo_root is not None:
        if str(repo_root) not in sys.path:
            sys.path.insert(0, str(repo_root))
        try:
            from common import get_api_key as shared_get_api_key
        except ImportError:
            pass
        else:
            get_api_key = shared_get_api_key

    if get_api_key is not None:
        for candidate in candidates:
            try:
                return get_api_key(candidate)
            except KeyError:
                pass

    for candidate in candidates:
        value = os.getenv(candidate, "").strip()
        if value:
            return value

    joined = ", ".join(candidates)
    raise RuntimeError(
        f"Missing API key. Set one of [{joined}] in the environment"
        " or the top-level config file."
    )

GEMINI_API_KEY = load_api_key("GEMINI_API_KEY")

MODEL = "gemini-2.5-flash"
ENDPOINT = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent?key={GEMINI_API_KEY}"

# Input images directory (recursively scanned)
DATASET_DIR = "/kaggle/input/diagram2graph-dataset/diagram2graph"  # <--- change this
# Output directory for per-image TTL files
OUTPUT_DIR = "/kaggle/working/ttl_fewshot_outputs"  # <--- change this
# Optional: also write a small manifest JSON with image->ttl mapping
WRITE_MANIFEST = True
MANIFEST_PATH = os.path.join(OUTPUT_DIR, "manifest.json")

VALID_IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}

# =========================
# 2) PROMPT (one-shot)
# =========================
PROMPT_TEXT = r"""

System
You are a diagram-to-graph extractor. Given a flowchart/diagram image, output ONLY valid Turtle using the diagram2graph (d2g) vocabulary.

Mandatory Instructions (follow exactly)
1) Prefixes (only these two)
@prefix d2g:  <http://example.org/diagram2graph#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

2) Identifiers (IRIs)
- Fixed patterns:
  Nodes: <http://example.org/diagram/diagram/node/{N}>
  Edges: <http://example.org/diagram/diagram/edge/{E}>
- Node numbering: visual reading order (top→bottom, left→right).
- Edge IDs (strict rule): build each edge IRI as ST, where S is the source node ID and T is the target node ID.
  Example: the edge between node/1 and node/2 MUST be <http://example.org/diagram/diagram/edge/12>

3) Nodes (create one node per readable shape)
For each shape:
<…/node/N> a d2g:Node, d2g:{Start|Process|Decision|Delay|Terminator} ;
    rdfs:label "exact text from the shape" ;
    d2g:shape d2g:{TartEvent|TaK|Gateway|Delay|EndEvent} .
Required shape mapping (use these exact dataset tokens—even if they look misspelled):
- Start/Begin (rounded oval) → d2g:TartEvent
- End/Stop   (rounded oval) → d2g:EndEvent
- Rectangle  (process)      → d2g:TaK
- Diamond    (decision)     → d2g:Gateway
- Delay      (delay symbol) → d2g:Delay

4) Edges (create a dedicated Edge resource per arrow)
For each arrow:
<…/edge/ST> a d2g:Edge, d2g:{Solid|Dashed} ;
    d2g:source <…/node/S> ;
    d2g:target <…/node/T> ;
    d2g:relationshipType d2g:{Follows|Branches} ;
    [optional] d2g:relationshipValue "label on the arrow (e.g., Yes/No)" .
Rules:
- Outgoing arrows from a Decision node must use relationshipType = d2g:Branches, and include relationshipValue if text exists (Yes/No/etc.).
- All other arrows use relationshipType = d2g:Follows.
- Set line style precisely: d2g:Solid for solid lines; d2g:Dashed for dotted/dashed lines.

5) Convenience node-to-node triples (also required)
- For Follows:  <…/node/S> d2g:follows  <…/node/T> .
- For Branches: <…/node/S> d2g:branches <…/node/T> .

6) Output rules
- Output Turtle only; no explanations or extra text.
- Group by subject; one triple per line; end every line with a period.
- Every edge’s source/target must reference existing node IRIs.
- Do NOT use any prefixes or vocabularies other than d2g and rdfs.
- Write class and shape tokens EXACTLY as specified above (no spelling changes).

Few-shot Examples (gold TTL from the dataset)
# Example A (from 3.png / 3.ttl)
@prefix d2g: <http://example.org/diagram2graph#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

# === Nodes ===
<http://example.org/diagram/diagram/node/1> a d2g:Start, d2g:Node ;
    rdfs:label "Begin" ;
    d2g:shape d2g:TartEvent .

<http://example.org/diagram/diagram/node/2> a d2g:Process, d2g:Node ;
    rdfs:label "SUM=0" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/3> a d2g:Process, d2g:Node ;
    rdfs:label "Input N" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/4> a d2g:Decision, d2g:Node ;
    rdfs:label "N!=0?" ;
    d2g:shape d2g:Gateway .

<http://example.org/diagram/diagram/node/5> a d2g:Process, d2g:Node ;
    rdfs:label "Print SUM" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/6> a d2g:Process, d2g:Node ;
    rdfs:label "REM=N%10" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/7> a d2g:Process, d2g:Node ;
    rdfs:label "SUM=SUM+REM" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/8> a d2g:Process, d2g:Node ;
    rdfs:label "N=N/10" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/9> a d2g:Terminator, d2g:Node ;
    rdfs:label "Stop" ;
    d2g:shape d2g:EndEvent .

# === Edges ===
<http://example.org/diagram/diagram/edge/12> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/1> ;
    d2g:target <http://example.org/diagram/diagram/node/2> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/1> d2g:follows <http://example.org/diagram/diagram/node/2> .

<http://example.org/diagram/diagram/edge/23> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/2> ;
    d2g:target <http://example.org/diagram/diagram/node/3> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/2> d2g:follows <http://example.org/diagram/diagram/node/3> .

<http://example.org/diagram/diagram/edge/34> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/3> ;
    d2g:target <http://example.org/diagram/diagram/node/4> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/3> d2g:follows <http://example.org/diagram/diagram/node/4> .

<http://example.org/diagram/diagram/edge/45> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/4> ;
    d2g:target <http://example.org/diagram/diagram/node/5> ;
    d2g:relationshipType d2g:Follows ;
    d2g:relationshipValue "False" .

<http://example.org/diagram/diagram/node/4> d2g:follows <http://example.org/diagram/diagram/node/5> .

<http://example.org/diagram/diagram/edge/46> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/4> ;
    d2g:target <http://example.org/diagram/diagram/node/6> ;
    d2g:relationshipType d2g:Follows ;
    d2g:relationshipValue "TRUE" .

<http://example.org/diagram/diagram/node/4> d2g:follows <http://example.org/diagram/diagram/node/6> .

<http://example.org/diagram/diagram/edge/67> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/6> ;
    d2g:target <http://example.org/diagram/diagram/node/7> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/6> d2g:follows <http://example.org/diagram/diagram/node/7> .

<http://example.org/diagram/diagram/edge/78> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/7> ;
    d2g:target <http://example.org/diagram/diagram/node/8> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/7> d2g:follows <http://example.org/diagram/diagram/node/8> .

<http://example.org/diagram/diagram/edge/84> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/8> ;
    d2g:target <http://example.org/diagram/diagram/node/4> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/8> d2g:follows <http://example.org/diagram/diagram/node/4> .

<http://example.org/diagram/diagram/edge/59> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/5> ;
    d2g:target <http://example.org/diagram/diagram/node/9> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/5> d2g:follows <http://example.org/diagram/diagram/node/9> .
# Example B (from 4.png / 4.ttl)
@prefix d2g: <http://example.org/diagram2graph#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

# === Nodes ===
<http://example.org/diagram/diagram/node/1> a d2g:Start, d2g:Node ;
    rdfs:label "Begin" ;
    d2g:shape d2g:TartEvent .

<http://example.org/diagram/diagram/node/2> a d2g:Process, d2g:Node ;
    rdfs:label "Initialize weights and biases With random values" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/3> a d2g:Process, d2g:Node ;
    rdfs:label "Perform oprations" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/4> a d2g:Process, d2g:Node ;
    rdfs:label "Calculate Error" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/5> a d2g:Decision, d2g:Node ;
    rdfs:label "Error<=Error" ;
    d2g:shape d2g:Gateway .

<http://example.org/diagram/diagram/node/6> a d2g:Decision, d2g:Node ;
    rdfs:label "Epoch>=Epoch" ;
    d2g:shape d2g:Gateway .

<http://example.org/diagram/diagram/node/7> a d2g:Process, d2g:Node ;
    rdfs:label "Update weights and biases" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/8> a d2g:Process, d2g:Node ;
    rdfs:label "Epoch=epoch+1" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/9> a d2g:Terminator, d2g:Node ;
    rdfs:label "End" ;
    d2g:shape d2g:EndEvent .

# === Edges ===
<http://example.org/diagram/diagram/edge/12> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/1> ;
    d2g:target <http://example.org/diagram/diagram/node/2> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/1> d2g:follows <http://example.org/diagram/diagram/node/2> .

<http://example.org/diagram/diagram/edge/23> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/2> ;
    d2g:target <http://example.org/diagram/diagram/node/3> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/2> d2g:follows <http://example.org/diagram/diagram/node/3> .

<http://example.org/diagram/diagram/edge/34> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/3> ;
    d2g:target <http://example.org/diagram/diagram/node/4> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/3> d2g:follows <http://example.org/diagram/diagram/node/4> .

<http://example.org/diagram/diagram/edge/45> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/4> ;
    d2g:target <http://example.org/diagram/diagram/node/5> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/4> d2g:follows <http://example.org/diagram/diagram/node/5> .

<http://example.org/diagram/diagram/edge/59> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/5> ;
    d2g:target <http://example.org/diagram/diagram/node/9> ;
    d2g:relationshipType d2g:Follows ;
    d2g:relationshipValue "Yes" .

<http://example.org/diagram/diagram/node/5> d2g:follows <http://example.org/diagram/diagram/node/9> .

<http://example.org/diagram/diagram/edge/56> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/5> ;
    d2g:target <http://example.org/diagram/diagram/node/6> ;
    d2g:relationshipType d2g:Follows ;
    d2g:relationshipValue "No" .

<http://example.org/diagram/diagram/node/5> d2g:follows <http://example.org/diagram/diagram/node/6> .

<http://example.org/diagram/diagram/edge/69> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/6> ;
    d2g:target <http://example.org/diagram/diagram/node/9> ;
    d2g:relationshipType d2g:Follows ;
    d2g:relationshipValue "Yes" .

<http://example.org/diagram/diagram/node/6> d2g:follows <http://example.org/diagram/diagram/node/9> .

<http://example.org/diagram/diagram/edge/67> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/6> ;
    d2g:target <http://example.org/diagram/diagram/node/7> ;
    d2g:relationshipType d2g:Follows ;
    d2g:relationshipValue "No" .

<http://example.org/diagram/diagram/node/6> d2g:follows <http://example.org/diagram/diagram/node/7> .

<http://example.org/diagram/diagram/edge/78> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/7> ;
    d2g:target <http://example.org/diagram/diagram/node/8> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/7> d2g:follows <http://example.org/diagram/diagram/node/8> .

<http://example.org/diagram/diagram/edge/83> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/8> ;
    d2g:target <http://example.org/diagram/diagram/node/3> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/8> d2g:follows <http://example.org/diagram/diagram/node/3> .
# Example C (from 6.png / 6.ttl)
@prefix d2g: <http://example.org/diagram2graph#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

# === Nodes ===
<http://example.org/diagram/diagram/node/1> a d2g:Start, d2g:Node ;
    rdfs:label "Start" ;
    d2g:shape d2g:TartEvent .

<http://example.org/diagram/diagram/node/2> a d2g:Decision, d2g:Node ;
    rdfs:label "Is target sector < max sector?" ;
    d2g:shape d2g:Gateway .

<http://example.org/diagram/diagram/node/3> a d2g:Process, d2g:Node ;
    rdfs:label "Increment Sector" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/4> a d2g:Process, d2g:Node ;
    rdfs:label "Target Sector=0" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/5> a d2g:Decision, d2g:Node ;
    rdfs:label "Want to update?" ;
    d2g:shape d2g:Gateway .

<http://example.org/diagram/diagram/node/6> a d2g:Process, d2g:Node ;
    rdfs:label "Increment head" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/7> a d2g:Process, d2g:Node ;
    rdfs:label "Target head=0" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/8> a d2g:Process, d2g:Node ;
    rdfs:label "Increment" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/9> a d2g:Terminator, d2g:Node ;
    rdfs:label "End" ;
    d2g:shape d2g:EndEvent .

# === Edges ===
<http://example.org/diagram/diagram/edge/12> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/1> ;
    d2g:target <http://example.org/diagram/diagram/node/2> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/1> d2g:follows <http://example.org/diagram/diagram/node/2> .

<http://example.org/diagram/diagram/edge/23> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/2> ;
    d2g:target <http://example.org/diagram/diagram/node/3> ;
    d2g:relationshipType d2g:Follows ;
    d2g:relationshipValue "Yes" .

<http://example.org/diagram/diagram/node/2> d2g:follows <http://example.org/diagram/diagram/node/3> .

<http://example.org/diagram/diagram/edge/39> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/3> ;
    d2g:target <http://example.org/diagram/diagram/node/9> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/3> d2g:follows <http://example.org/diagram/diagram/node/9> .

<http://example.org/diagram/diagram/edge/24> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/2> ;
    d2g:target <http://example.org/diagram/diagram/node/4> ;
    d2g:relationshipType d2g:Follows ;
    d2g:relationshipValue "No" .

<http://example.org/diagram/diagram/node/2> d2g:follows <http://example.org/diagram/diagram/node/4> .

<http://example.org/diagram/diagram/edge/45> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/4> ;
    d2g:target <http://example.org/diagram/diagram/node/5> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/4> d2g:follows <http://example.org/diagram/diagram/node/5> .

<http://example.org/diagram/diagram/edge/56> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/5> ;
    d2g:target <http://example.org/diagram/diagram/node/6> ;
    d2g:relationshipType d2g:Follows ;
    d2g:relationshipValue "Yes" .

<http://example.org/diagram/diagram/node/5> d2g:follows <http://example.org/diagram/diagram/node/6> .

<http://example.org/diagram/diagram/edge/69> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/6> ;
    d2g:target <http://example.org/diagram/diagram/node/9> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/6> d2g:follows <http://example.org/diagram/diagram/node/9> .

<http://example.org/diagram/diagram/edge/57> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/5> ;
    d2g:target <http://example.org/diagram/diagram/node/7> ;
    d2g:relationshipType d2g:Follows ;
    d2g:relationshipValue "No" .

<http://example.org/diagram/diagram/node/5> d2g:follows <http://example.org/diagram/diagram/node/7> .

<http://example.org/diagram/diagram/edge/78> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/7> ;
    d2g:target <http://example.org/diagram/diagram/node/8> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/7> d2g:follows <http://example.org/diagram/diagram/node/8> .

<http://example.org/diagram/diagram/edge/89> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/8> ;
    d2g:target <http://example.org/diagram/diagram/node/9> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/8> d2g:follows <http://example.org/diagram/diagram/node/9> .

```

Example B — (image similar to 4.png)
Assistant (expected TTL)
```turtle
@prefix d2g: <http://example.org/diagram2graph#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

# === Nodes ===
<http://example.org/diagram/diagram/node/1> a d2g:Start, d2g:Node ;
    rdfs:label "Begin" ;
    d2g:shape d2g:TartEvent .

<http://example.org/diagram/diagram/node/2> a d2g:Process, d2g:Node ;
    rdfs:label "Initialize weights and biases With random values" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/3> a d2g:Process, d2g:Node ;
    rdfs:label "Perform oprations" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/4> a d2g:Process, d2g:Node ;
    rdfs:label "Calculate Error" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/5> a d2g:Decision, d2g:Node ;
    rdfs:label "Error<=Error" ;
    d2g:shape d2g:Gateway .

<http://example.org/diagram/diagram/node/6> a d2g:Decision, d2g:Node ;
    rdfs:label "Epoch>=Epoch" ;
    d2g:shape d2g:Gateway .

<http://example.org/diagram/diagram/node/7> a d2g:Process, d2g:Node ;
    rdfs:label "Update weights and biases" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/8> a d2g:Process, d2g:Node ;
    rdfs:label "Epoch=epoch+1" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/9> a d2g:Terminator, d2g:Node ;
    rdfs:label "End" ;
    d2g:shape d2g:EndEvent .

# === Edges ===
<http://example.org/diagram/diagram/edge/12> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/1> ;
    d2g:target <http://example.org/diagram/diagram/node/2> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/1> d2g:follows <http://example.org/diagram/diagram/node/2> .

<http://example.org/diagram/diagram/edge/23> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/2> ;
    d2g:target <http://example.org/diagram/diagram/node/3> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/2> d2g:follows <http://example.org/diagram/diagram/node/3> .

<http://example.org/diagram/diagram/edge/34> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/3> ;
    d2g:target <http://example.org/diagram/diagram/node/4> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/3> d2g:follows <http://example.org/diagram/diagram/node/4> .

<http://example.org/diagram/diagram/edge/45> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/4> ;
    d2g:target <http://example.org/diagram/diagram/node/5> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/4> d2g:follows <http://example.org/diagram/diagram/node/5> .

<http://example.org/diagram/diagram/edge/59> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/5> ;
    d2g:target <http://example.org/diagram/diagram/node/9> ;
    d2g:relationshipType d2g:Follows ;
    d2g:relationshipValue "Yes" .

<http://example.org/diagram/diagram/node/5> d2g:follows <http://example.org/diagram/diagram/node/9> .

<http://example.org/diagram/diagram/edge/56> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/5> ;
    d2g:target <http://example.org/diagram/diagram/node/6> ;
    d2g:relationshipType d2g:Follows ;
    d2g:relationshipValue "No" .

<http://example.org/diagram/diagram/node/5> d2g:follows <http://example.org/diagram/diagram/node/6> .

<http://example.org/diagram/diagram/edge/69> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/6> ;
    d2g:target <http://example.org/diagram/diagram/node/9> ;
    d2g:relationshipType d2g:Follows ;
    d2g:relationshipValue "Yes" .

<http://example.org/diagram/diagram/node/6> d2g:follows <http://example.org/diagram/diagram/node/9> .

<http://example.org/diagram/diagram/edge/67> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/6> ;
    d2g:target <http://example.org/diagram/diagram/node/7> ;
    d2g:relationshipType d2g:Follows ;
    d2g:relationshipValue "No" .

<http://example.org/diagram/diagram/node/6> d2g:follows <http://example.org/diagram/diagram/node/7> .

<http://example.org/diagram/diagram/edge/78> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/7> ;
    d2g:target <http://example.org/diagram/diagram/node/8> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/7> d2g:follows <http://example.org/diagram/diagram/node/8> .

<http://example.org/diagram/diagram/edge/83> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/8> ;
    d2g:target <http://example.org/diagram/diagram/node/3> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/8> d2g:follows <http://example.org/diagram/diagram/node/3> .

```

Example C — (image similar to 6.png)
Assistant (expected TTL)
```turtle
@prefix d2g: <http://example.org/diagram2graph#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

# === Nodes ===
<http://example.org/diagram/diagram/node/1> a d2g:Start, d2g:Node ;
    rdfs:label "Start" ;
    d2g:shape d2g:TartEvent .

<http://example.org/diagram/diagram/node/2> a d2g:Decision, d2g:Node ;
    rdfs:label "Is target sector < max sector?" ;
    d2g:shape d2g:Gateway .

<http://example.org/diagram/diagram/node/3> a d2g:Process, d2g:Node ;
    rdfs:label "Increment Sector" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/4> a d2g:Process, d2g:Node ;
    rdfs:label "Target Sector=0" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/5> a d2g:Decision, d2g:Node ;
    rdfs:label "Want to update?" ;
    d2g:shape d2g:Gateway .

<http://example.org/diagram/diagram/node/6> a d2g:Process, d2g:Node ;
    rdfs:label "Increment head" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/7> a d2g:Process, d2g:Node ;
    rdfs:label "Target head=0" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/8> a d2g:Process, d2g:Node ;
    rdfs:label "Increment" ;
    d2g:shape d2g:TaK .

<http://example.org/diagram/diagram/node/9> a d2g:Terminator, d2g:Node ;
    rdfs:label "End" ;
    d2g:shape d2g:EndEvent .

# === Edges ===
<http://example.org/diagram/diagram/edge/12> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/1> ;
    d2g:target <http://example.org/diagram/diagram/node/2> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/1> d2g:follows <http://example.org/diagram/diagram/node/2> .

<http://example.org/diagram/diagram/edge/23> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/2> ;
    d2g:target <http://example.org/diagram/diagram/node/3> ;
    d2g:relationshipType d2g:Follows ;
    d2g:relationshipValue "Yes" .

<http://example.org/diagram/diagram/node/2> d2g:follows <http://example.org/diagram/diagram/node/3> .

<http://example.org/diagram/diagram/edge/39> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/3> ;
    d2g:target <http://example.org/diagram/diagram/node/9> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/3> d2g:follows <http://example.org/diagram/diagram/node/9> .

<http://example.org/diagram/diagram/edge/24> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/2> ;
    d2g:target <http://example.org/diagram/diagram/node/4> ;
    d2g:relationshipType d2g:Follows ;
    d2g:relationshipValue "No" .

<http://example.org/diagram/diagram/node/2> d2g:follows <http://example.org/diagram/diagram/node/4> .

<http://example.org/diagram/diagram/edge/45> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/4> ;
    d2g:target <http://example.org/diagram/diagram/node/5> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/4> d2g:follows <http://example.org/diagram/diagram/node/5> .

<http://example.org/diagram/diagram/edge/56> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/5> ;
    d2g:target <http://example.org/diagram/diagram/node/6> ;
    d2g:relationshipType d2g:Follows ;
    d2g:relationshipValue "Yes" .

<http://example.org/diagram/diagram/node/5> d2g:follows <http://example.org/diagram/diagram/node/6> .

<http://example.org/diagram/diagram/edge/69> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/6> ;
    d2g:target <http://example.org/diagram/diagram/node/9> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/6> d2g:follows <http://example.org/diagram/diagram/node/9> .

<http://example.org/diagram/diagram/edge/57> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/5> ;
    d2g:target <http://example.org/diagram/diagram/node/7> ;
    d2g:relationshipType d2g:Follows ;
    d2g:relationshipValue "No" .

<http://example.org/diagram/diagram/node/5> d2g:follows <http://example.org/diagram/diagram/node/7> .

<http://example.org/diagram/diagram/edge/78> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/7> ;
    d2g:target <http://example.org/diagram/diagram/node/8> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/7> d2g:follows <http://example.org/diagram/diagram/node/8> .

<http://example.org/diagram/diagram/edge/89> a d2g:Solid, d2g:Edge ;
    d2g:source <http://example.org/diagram/diagram/node/8> ;
    d2g:target <http://example.org/diagram/diagram/node/9> ;
    d2g:relationshipType d2g:Follows .

<http://example.org/diagram/diagram/node/8> d2g:follows <http://example.org/diagram/diagram/node/9> .

```

---
User
Here is a new image. Produce the TTL now.


"""

# =========================
# 3) HELPERS
# =========================
def get_mime_type(path: str) -> str:
    ext = Path(path).suffix.lower()
    if ext in {".jpg", ".jpeg"}:
        return "image/jpeg"
    if ext == ".png":
        return "image/png"
    if ext in {".tif", ".tiff"}:
        return "image/tiff"
    if ext == ".webp":
        return "image/webp"
    if ext == ".bmp":
        return "image/bmp"
    return "image/jpeg"

def encode_image_to_base64(image_path: str) -> Optional[str]:
    try:
        # Ensure the image is readable; PIL open also validates file integrity
        with Image.open(image_path) as _:
            pass
        data = Path(image_path).read_bytes()
        return base64.b64encode(data).decode("utf-8")
    except FileNotFoundError:
        print(f"[WARN] File not found: {image_path}")
        return None
    except Exception as e:
        print(f"[WARN] Failed to read {image_path}: {e}")
        return None

def build_request(image_b64: str, mime_type: str, prompt: str) -> dict:
    return {
        "contents": [
            {
                "role": "user",
                "parts": [
                    {"inlinedata": {"mimeType": mime_type, "data": image_b64}},
                    {"text": prompt},
                ],
            }
        ],
        "generationConfig": {
            "temperature": 0.1,
            "topP": 0.9,
            "maxOutputTokens": 8192,
        },
    }

def _strip_code_fences(text: str) -> str:
    """
    Removes ```turtle ... ``` or ``` ... ``` fences if present.
    """
    t = text.strip()
    if t.startswith("```"):
        first_newline = t.find("\n")
        if first_newline != -1 and t[:first_newline].startswith("```"):
            t = t[first_newline + 1 :]
        if t.endswith("```"):
            t = t[:-3].rstrip()
    return t.strip()

def call_gemini(payload: dict, max_retries: int = 4) -> Optional[str]:
    """
    Calls the Gemini API with basic retry/backoff on 429/5xx.
    Returns clean Turtle (TTL) text, or None.
    """
    headers = {"Content-Type": "application/json"}
    backoff = 2.0
    for attempt in range(max_retries):
        try:
            resp = requests.post(ENDPOINT, headers=headers, data=json.dumps(payload), timeout=120)
            if resp.status_code >= 500 or resp.status_code == 429:
                print(f"[INFO] Retryable HTTP {resp.status_code}; attempt {attempt+1}/{max_retries}")
                time.sleep(backoff)
                backoff *= 1.8
                continue
            resp.raise_for_status()
            data = resp.json()
            if "candidates" in data and data["candidates"]:
                raw = data["candidates"][0]["content"]["parts"][0]["text"]
                return _strip_code_fences(raw)
            return None
        except requests.exceptions.RequestException as e:
            print(f"[WARN] Request error: {e}; attempt {attempt+1}/{max_retries}")
            time.sleep(backoff)
            backoff *= 1.8
    return None

def ensure_dir(path: str):
    Path(path).mkdir(parents=True, exist_ok=True)

# =========================
# 4) MAIN: per-image .ttl files
# =========================
def main():
    ensure_dir(OUTPUT_DIR)

    manifest = []  # optional record of what we wrote

    for root, _, files in os.walk(DATASET_DIR):
        for fname in files:
            ext = Path(fname).suffix.lower()
            if ext not in VALID_IMAGE_EXTS:
                continue

            img_path = os.path.join(root, fname)
            stem = Path(fname).stem  # "1.png" -> "1"
            ttl_path = os.path.join(OUTPUT_DIR, f"{stem}.ttl")

            print(f"[INFO] Processing: {img_path} -> {ttl_path}")

            image_b64 = encode_image_to_base64(img_path)
            if not image_b64:
                print(f"[SKIP] Could not encode {img_path}")
                continue

            mime = get_mime_type(img_path)
            payload = build_request(image_b64, mime, PROMPT_TEXT)
            ttl_text = call_gemini(payload)

            if not ttl_text:
                print(f"[WARN] No TTL returned for {img_path}")
                continue

            # Write the TTL file
            with open(ttl_path, "w", encoding="utf-8") as f:
                f.write(ttl_text if ttl_text.endswith("\n") else ttl_text + "\n")

            manifest.append({"source_image": img_path, "ttl_file": ttl_path})

    if WRITE_MANIFEST:
        ensure_dir(OUTPUT_DIR)
        with open(MANIFEST_PATH, "w", encoding="utf-8") as mf:
            json.dump({"items": manifest}, mf, indent=2, ensure_ascii=False)

    print(f"[DONE] Wrote {len(manifest)} TTL files to: {OUTPUT_DIR}")
    if WRITE_MANIFEST:
        print(f"[INFO] Manifest: {MANIFEST_PATH}")

if __name__ == "__main__":
    main()


In [ ]:
!zip -r /kaggle/working/fewshot_outputs.zip /kaggle/working/ttl_fewshot_outputs